# Genetic similarities among the study dogs

The main question is whether each breed's study group contains two or more distinct genetic groups that recur across methods and settings and could reflect different ancestry or families. Such groups could complicate comparisons of training outcomes. We assess this question within each breed; differences between the two recorded breeds are expected in the combined view.

Dogs can also be related without forming separate visible clusters. An absence of clear clusters would not establish that the dogs are unrelated.

We examined all 164 dogs in the broader study group once each: 116 German Shorthaired Pointers and 48 Labrador Retrievers. The dogs in the main comparison are included in this group. We looked at both breeds together and then examined each breed separately.

This chapter uses three ways of drawing genetic similarities: principal component analysis (PCA), UMAP, and t-SNE. Each point is a dog. The drawings describe the data; they do not establish a dog's identity or test whether a genetic region influences training outcomes.

## Running this chapter

This chapter combines the scientific account with executable analysis steps. Run the cells in order with a Python kernel containing the project's structure-analysis dependencies. The chapter needs the authorized QC datasets from Notebook 02 and PLINK 1.9. Those private genotypes are not distributed with the public repository.

The code below runs the complete structure analysis: marker pruning, PCA, UMAP, t-SNE, alternative layouts, and within-breed genetic sharing. Results are saved separately from the original analysis. The summary table and scientific conclusions describe the original study; they remain written directly in the notebook and should be reviewed if the data or settings change.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

study_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/genetic_structure.py").exists())
run_name = "version2-preview" if os.name == "nt" else "version2-preview-linux"
run_folder = study_root / "data/structure" / run_name
figure_folder = study_root / "results/private/03_structure" / run_name / "figures"
qc_file = study_root / "data/qc/manifest.private.json"

# Set this to your native PLINK executable if it is not already configured.
plink_executable = None

### Read the prepared data and locate PLINK

We use the broader study groups once, so the main-comparison dogs are included without duplicating them. This reads the QC data locations rather than rebuilding the study groups. On Windows, the saved PLINK installation can be reused; on Linux or WSL, select a native Linux PLINK executable.

In [ ]:
qc = json.loads(qc_file.read_text(encoding="utf-8"))
assert qc["status"] == "complete", "Complete Notebook 02 first."
genotype_prefixes = {
    dataset["breed"]: next(study_root / p.replace(chr(92), "/") for p in dataset["files"]
                           if p.endswith(".fam")).with_suffix("")
    for dataset in qc["datasets"]
    if dataset["scenario"] == "not-behelim-vs-behelim"
}
saved_environment = study_root / "data/processed/private/plink_environment.private.json"
if plink_executable is None and os.name == "nt" and saved_environment.exists():
    plink_executable = json.loads(saved_environment.read_text(encoding="utf-8"))["executable"]
local_plink = study_root / "data/tools" / (
    "plink-1.9.0-20260927-win64/plink.exe" if os.name == "nt"
    else "plink-1.9.0-20260927-linux64/plink"
)
plink_executable = plink_executable or (str(local_plink) if local_plink.is_file() else shutil.which("plink"))
if not plink_executable or not Path(plink_executable).is_file():
    raise FileNotFoundError("Install the project tool with: python scripts/setup_plink.py (from the repository), or set plink_executable.")
version = subprocess.check_output([str(plink_executable), "--version"], text=True)
assert "PLINK v1.9" in version, "This chapter uses PLINK 1.9."
print(version.strip())
print("QC data located for:", ", ".join(sorted(genotype_prefixes)))

## Three views of the same dogs

PCA summarizes the largest patterns of variation along a series of axes, called principal components. The first two axes provide a simple view of the strongest patterns, but they represent only part of the variation among the dogs.

UMAP and t-SNE emphasize similarities among nearby dogs and can reveal patterns that are less obvious in a two-axis PCA plot. Their appearance depends on the settings used. The gaps between apparent groups do not directly measure genetic distance, and the size of a drawn group does not measure its diversity.

We reduced redundancy among strongly correlated markers before drawing these views. Within each view, all three methods use the same dogs and marker information. UMAP and t-SNE were built from the first 20 principal components; outcomes were not used to arrange the points. Colors were added afterwards for interpretation.

| View | Dogs | Markers after reducing redundancy | Variation represented by the first two PCs |
| --- | ---: | ---: | ---: |
| Both breeds | 164 | 2,370 | 13.8% |
| German Shorthaired Pointer | 116 | 16,053 | 13.3% |
| Labrador Retriever | 48 | 17,268 | 15.5% |

## Generate the structure results

The script applies a 50-marker pruning window, moves it by five markers, and removes redundancy above a squared correlation of 0.2. It calculates PCA within each breed and for their shared pruned markers. UMAP and t-SNE use the first 20 components weighted by their variation, with random seed 20261002; additional layouts use seed 42. Relatedness is estimated within each breed.

The following cell runs [the analysis script](../scripts/genetic_structure.py) using this notebook's Python environment. Progress appears beneath the cell. A completed preview run is reused when its analysis script and QC manifest match; choose another `run_name` above for a fresh analysis. Version 1 and the original result files are not overwritten.

In [ ]:
run_folder.mkdir(parents=True, exist_ok=True)
environment_file = run_folder / "plink_environment.private.json"
environment = {"executable": str(Path(plink_executable).resolve()),
               "executable_sha256": hashlib.sha256(Path(plink_executable).read_bytes()).hexdigest()}
script = study_root / "scripts/genetic_structure.py"
manifest_file = run_folder / "manifest.private.json"
if manifest_file.exists():
    manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
    assert manifest["status"] == "complete"
    assert manifest["script_sha256"] == hashlib.sha256(script.read_bytes()).hexdigest(), "Choose a new run_name for changed code."
    assert manifest["qc_manifest_sha256"] == hashlib.sha256(qc_file.read_bytes()).hexdigest(), "Choose a new run_name for changed QC inputs."
    print("Using the completed preview run.")
else:
    from importlib.util import find_spec
    required = ["numpy", "matplotlib", "sklearn", "threadpoolctl", "umap", "nbformat"]
    missing = [name for name in required if find_spec(name) is None]
    if missing:
        raise ModuleNotFoundError(
            "Rerunning the analysis needs: " + ", ".join(missing)
            + ". Install scripts/requirements-structure.txt into this notebook's kernel with %pip."
        )
    environment_file.write_text(json.dumps(environment, indent=2), encoding="utf-8")
    command = [sys.executable, str(script), "--root", str(study_root),
               "--environment", str(environment_file), "--output", str(run_folder),
               "--figures", str(figure_folder)]
    subprocess.run(command, cwd=study_root, check=True)
    manifest = json.loads(manifest_file.read_text(encoding="utf-8"))

### Load the generated figures

This small function displays a saved figure wherever it is discussed in the chapter. The figure files remain available as PNG and SVG for manuscript preparation.

In [ ]:
from IPython.display import Image, display

def display_figure(name):
    path = figure_folder / f"{name}.png"
    relative = path.relative_to(study_root).as_posix()
    figure_hashes = {key.replace(chr(92), "/"): value for key, value in manifest["figures"].items()}
    expected = figure_hashes[relative]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected
    display(Image(filename=str(path)))

### Overview

The top row colors dogs by recorded breed. The other rows show each breed separately, with colors indicating training-outcome groups. Across the combined views, many dogs group with others of the same breed, but the groups overlap. Within each breed, the PCA plots show some dogs away from the largest concentration of points. We should check whether the same dogs form separate groups across methods and alternative settings before calling these distinct populations or families. Visual inspection alone does not establish statistically significant structure, identify another breed, or justify removing a dog.

In [ ]:
display_figure("structure_overview")

### Both breeds together

The combined view uses markers retained in both breeds and selected after reducing correlated markers within each breed. It is a descriptive comparison of the study dogs, rather than a reference-based test of ancestry. Breed labels come from the accepted January records.

In [ ]:
display_figure("combined_structure")

### German Shorthaired Pointers

Behavioral cases occur in more than one part of these plots, alongside controls. The maps do not supply an association test. Family relationships, population history, and the small case group all need consideration before interpreting a visual difference.

In [ ]:
display_figure("GSP_structure")

### Labrador Retrievers

The Labrador comparison contains only seven behavioral cases. The positions of a few cases in a drawing cannot establish a genetic explanation for elimination. We will retain the distinction between descriptive structure and the later, predefined replication tests.

In [ ]:
display_figure("LR_structure")

## How much information is visible?

This figure shows how much variation each of the first 20 principal components represents. The first 20 together account for 37.1% in the combined analysis, 41.9% in GSP, and 63.7% in LR. The nonlinear maps therefore summarize this selected part of the genetic information, rather than every difference in the genotype data.

### From principal components to map inputs

The function below shows the same PCA-score calculation as the analysis script. It reads the saved results directly, so viewing them does not require importing the full analysis or its UMAP dependency. It reads the PCA components and multiplies each by the square root of its eigenvalue, so components explaining more genetic variation contribute more to the nonlinear maps. We show this step because it makes the relationship between PCA, UMAP, and t-SNE explicit.

In [ ]:
import numpy as np

def load_pca_scores(prefix, components=20):
    """Read PLINK components and weight each by its eigenvalue's square root."""
    with prefix.with_suffix('.eigenvec').open(encoding='utf-8') as handle:
        rows = [line.split() for line in handle if line.strip()]
    records = [dict(zip(rows[0], row)) for row in rows[1:]]
    eigenvalues=np.loadtxt(prefix.with_suffix('.eigenval'))
    assert np.isfinite(eigenvalues).all() and np.all(eigenvalues>=-1e-7)
    eigenvalues=np.maximum(eigenvalues,0)
    if components>len(eigenvalues):
        raise ValueError('Requested more components than the PCA provides')
    vectors=np.array([[float(row[f'PC{k+1}']) for k in range(components)] for row in records])
    scores=vectors*np.sqrt(eigenvalues[:components])
    assert np.isfinite(scores).all()
    return [row['IID'] for row in records], scores, eigenvalues

In [ ]:
dog_ids, map_inputs, eigenvalues = load_pca_scores(run_folder / "LR/03_pca")
assert map_inputs.shape == (manifest["summary"]["LR"]["dogs"], 20)
print(f"Labrador map inputs: {map_inputs.shape[0]} dogs, {map_inputs.shape[1]} components.")

In [ ]:
display_figure("pca_variation")

## Do the nonlinear drawings depend on their settings?

We repeated UMAP and t-SNE with alternative neighborhood settings and another random seed. The following figures show how their appearance changes. A consistent local neighborhood is more informative than a gap or cluster that appears in only one layout. Colors have the same meanings as in the main figures.

### Both breeds: alternative layouts

In [ ]:
display_figure("combined_layout_sensitivity")

### German Shorthaired Pointers: alternative layouts

In [ ]:
display_figure("GSP_layout_sensitivity")

### Labrador Retrievers: alternative layouts

In [ ]:
display_figure("LR_layout_sensitivity")

The Labrador layouts change substantially with the neighborhood setting. We should therefore be particularly cautious about naming discrete Labrador groups from these maps. We also checked how well each two-dimensional map preserves nearby dogs from the 20-component input. This check describes the drawing's fidelity, not whether a biological interpretation is correct.

### Check the completed analysis

Before interpreting the plots, this cell checks that the QC inputs still match the files used in the analysis and that all three views have recorded results. The script also checks sample membership, finite coordinates, and preservation of its input files. Detailed scientific verification of the original study is documented in the [structure methods note](../docs/methods/03%20Genetic%20structure%20and%20notebook%20figures.md).

In [ ]:
assert manifest["status"] == "complete"
assert manifest["source_qc_hashes_unchanged"]
for dataset in qc["datasets"]:
    for relative_path, expected in dataset["files"].items():
        assert hashlib.sha256((study_root / relative_path.replace(chr(92), "/")).read_bytes()).hexdigest() == expected
assert set(manifest["summary"]) == {"combined", "GSP", "LR"}
print("QC inputs verified; combined and both within-breed analyses completed.")

## Genetic sharing between pairs of dogs

We also estimated genetic sharing within each breed using PLINK. The horizontal axis gives the estimated fraction of genetic material shared by descent; the vertical axis counts pairs of dogs. The estimates suggest substantial relatedness in parts of both cohorts.

There are 124 GSP pairs and 26 LR pairs with estimates of at least 0.25. These estimates depend on the sampled population and should not be read as confirmed pedigree relationships. No pair had an estimate of 0.90 or higher, but this does not independently verify every sample identity. We have not removed dogs on the basis of these estimates.

In [ ]:
display_figure("within_breed_relatedness")

## Implications for the association analyses

Our aim is to determine whether reproducible, distinct groups are present within either breed. The current drawings do not establish distinct breed or family subgroups, and the separated points in PCA warrant review across the other views. Without reference dogs or pedigrees, we cannot assign ancestry or family membership from these maps.

The genetic-sharing estimates support accounting for relatedness even if this review finds no clear clusters. In a mixed model, a genetic relationship matrix represents similarities between every pair of dogs. Adjustment variables such as principal components describe broader patterns; they do not replace that relationship matrix. GEMMA can use both. With only seven Labrador cases and nine GSP cases, we still need to evaluate how well the chosen test behaves for these small binary-outcome comparisons before selecting the final model.

PCA may provide useful adjustment variables, but the number of components should be chosen with the small cohort sizes in mind. UMAP and t-SNE remain descriptive figures and will not automatically become model variables or exclusion rules. We have not selected a model by looking for stronger associations, and no association testing has been performed.